# ==========================================================
# FINAL 
# MP3 -> Transcript (Whisper) -> High-quality Minutes (Ollama)
# - No console streaming token prints
# - Stronger prompt for council/board style meetings
# - Descriptive, structured, and clean Markdown
# - Renders in Jupyter + saves .md file
# ==========================================================

## ---- Import Section ----

In [1]:
import torch                         # torch = PyTorch library (used here for GPU/CPU detection + dtype like float16/float32)
import librosa                       # librosa = audio loading/processing library (loads MP3 into arrays)
from transformers import pipeline    # pipeline = easy Hugging Face wrapper (we use it for Whisper ASR transcription)
from openai import OpenAI            # OpenAI client (we use it with Ollama's OpenAI-compatible API)
from IPython.display import display, Markdown  # display/Markdown = show Markdown output nicely inside Jupyter Notebook

## -----------------------------
## 0) SETTINGS (change only these)
## -----------------------------


In [2]:
# audio_file_path:
# - "r" before the string means "raw string"
# - raw string avoids issues with backslashes in Windows paths (\)
audio_file_path = r"D:\work\llm_engineering\Data\denver_extract.mp3"  # path to your input MP3 file

# whisper_model_name:
# - Whisper model for speech-to-text (ASR)
# - base.en is a good balance; tiny.en is faster but less accurate
whisper_model_name = "openai/whisper-base.en"

# minutes_model_name:
# - the LLM you will call via Ollama
minutes_model_name = "gpt-oss:120b-cloud"

# max_transcript_characters:
# - limits transcript length to keep LLM generation stable and faster
max_transcript_characters = 14000

# chunk_length_seconds:
# - Whisper will process audio in chunks of this many seconds
# - helps with long audio and memory usage
chunk_length_seconds = 45

## ==========================================================
## 1) GPU or CPU?
## ==========================================================

In [3]:
# is_gpu_available:
# - torch.cuda.is_available() returns True if CUDA GPU is available, else False
is_gpu_available = torch.cuda.is_available()

# device_number:
# - transformers pipeline uses:
#   device=0  -> first GPU (cuda:0)
#   device=-1 -> CPU
device_number = 0 if is_gpu_available else -1  # "if GPU then 0 else -1"

# Print whether CUDA is available
print("CUDA available:", is_gpu_available)

# If GPU exists, print GPU name (example: "NVIDIA GeForce GTX 1650")
if is_gpu_available:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce GTX 1650


## ==========================================================
## 2) Load audio (Whisper expects 16kHz mono)
## ==========================================================

In [4]:
print("\n[1/3] Loading audio...")  # \n makes a blank line before the text

# librosa.load(...) returns TWO things:
# 1) audio_data = waveform array (numbers representing sound amplitude)
# 2) sample_rate = samples per second
#
# We force:
# - sr=16000   -> resample audio to 16kHz (Whisper expects 16kHz)
# - mono=True  -> convert to mono (single channel)
audio_data, sample_rate = librosa.load(audio_file_path, sr=16000, mono=True)

# len(audio_data) = number of samples
# sample_rate = samples per second
# len(audio_data)/sample_rate = total seconds
print("Audio length (sec):", round(len(audio_data) / sample_rate, 1))  # round(..., 1) = 1 decimal place


[1/3] Loading audio...
Audio length (sec): 900.0


## ==========================================================
## 3) Transcribe with Whisper (with timestamps)
## ==========================================================

In [5]:
print("[2/3] Transcribing with Whisper...")

# speech_to_text_model = pipeline(...):
# - task: "automatic-speech-recognition" = speech to text
# - model: whisper_model_name
# - device: GPU/CPU selection
# - torch_dtype:
#     float16 on GPU (faster + less memory)
#     float32 on CPU (safe/default)
# - return_timestamps=True:
#     asks Whisper to return timestamps for parts of transcript
# - chunk_length_s:
#     audio is processed in segments
# - stride_length_s=(2,2):
#     overlap between chunks (left and right) to reduce cut-off words
speech_to_text_model = pipeline(
    "automatic-speech-recognition",              # task name
    model=whisper_model_name,                    # whisper model ID
    device=device_number,                        # 0 for GPU, -1 for CPU
    torch_dtype=torch.float16 if is_gpu_available else torch.float32,  # pick dtype based on GPU availability
    return_timestamps=True,                      # get timestamps in output
    chunk_length_s=chunk_length_seconds,         # chunk size in seconds
    stride_length_s=(2, 2),                      # overlap in seconds (left, right)
)

# transcription_result:
# - calling the pipeline like a function runs ASR
# - input is audio_data (waveform array)
transcription_result = speech_to_text_model(audio_data)

# convert_seconds_to_mmss(seconds_value):
# - converts time in seconds to "MM:SS"
# - handles tuple/list timestamps too (sometimes timestamp is (start,end))
def convert_seconds_to_mmss(seconds_value):
    """Convert seconds -> MM:SS (handles tuple timestamps too)."""

    # If timestamp is like (start, end), take the start time
    if isinstance(seconds_value, (tuple, list)):  # isinstance checks the type
        seconds_value = seconds_value[0]

    # If timestamp is missing (None), return placeholder
    if seconds_value is None:
        return "??:??"

    # Convert to integer seconds (remove decimals)
    seconds_value = int(seconds_value)

    # divmod(a, b) returns (a//b, a%b)
    # Here: minutes = seconds_value//60, seconds = seconds_value%60
    minutes, seconds = divmod(seconds_value, 60)

    # f-string formatting:
    # {minutes:02d} means "2 digits, pad with 0"
    return f"{minutes:02d}:{seconds:02d}"


# Build timestamped transcript (one line per chunk)
# Whisper pipeline can return output as:
# - transcription_result["chunks"] : list of chunk objects with "text" and "timestamp"
# OR
# - transcription_result["text"]   : plain text string (no chunks)
if "chunks" in transcription_result and transcription_result["chunks"]:
    transcript_lines = []  # list to store each transcript line
    previous_line = ""     # used to avoid adding duplicate lines

    # Loop over each chunk produced by Whisper
    for chunk in transcription_result["chunks"]:
        # chunk.get("text") returns the chunk text if it exists, else None
        # ( ... or "" ) ensures we never call .strip() on None
        text_part = (chunk.get("text") or "").strip()  # strip removes leading/trailing whitespace

        # If chunk has no text, skip it
        if not text_part:
            continue

        # chunk.get("timestamp") gives time info
        # convert_seconds_to_mmss turns it into [MM:SS]
        formatted_line = f"[{convert_seconds_to_mmss(chunk.get('timestamp'))}] {text_part}"

        # Avoid repeating the exact same line twice
        if formatted_line != previous_line:
            transcript_lines.append(formatted_line)  # add line to list
            previous_line = formatted_line           # update last seen line

    # Join all lines into one big transcript string separated by newline
    full_transcript = "\n".join(transcript_lines)

else:
    # If no chunks exist, just use the normal "text" field
    full_transcript = transcription_result.get("text", "")

# Trim transcript to max_transcript_characters:
# - keeps it from being too long
# - helps speed and reduces model confusion
full_transcript = full_transcript[:max_transcript_characters]

# Show how many characters we will send to LLM
print("Transcription done. Characters used:", len(full_transcript))

[2/3] Transcribing with Whisper...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> ha

Transcription done. Characters used: 9377


## ==========================================================
## 4) Generate Meeting Minutes via Ollama (PRO Prompt, no prints)
## ==========================================================

In [6]:

print("[3/3] Generating meeting minutes via Ollama...\n")

# ollama_client:
# - OpenAI client object
# - base_url points to local Ollama OpenAI-compatible server
# - api_key is required by client but Ollama ignores it (placeholder)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # placeholder for Ollama
)

# system_prompt:
# - system message tells the model "who it is" and strict rules
# - this heavily controls formatting + quality
system_prompt = """
You are an expert corporate secretary and council/board meeting scribe.

Your job:
Turn the transcript into clear, accurate, professional meeting minutes.

Output rules (STRICT):
- Output ONLY Markdown minutes (no preface, no explanation, no transcript)
- Do NOT copy transcript lines verbatim; rewrite and summarize (3-4 sentences)
- Keep facts faithful; do NOT invent details
- If something is unclear/missing, write 'Not specified'
- If no action items are explicitly assigned, write 'None mentioned'
- Use blank lines between sections
- Prefer concise bullets; use short paragraphs only where requested

Quality requirements:
- Write a descriptive Meeting Summary (6–8 sentences) explaining context, purpose, and outcomes
- Agenda should be inferred as best as possible from transcript
- Key Discussion Points should be grouped by topic, each topics should have 2–4 bullet points explanation.
- Each bullet should include a timestamp like [MM:SS] when relevant
- Decisions must be explicit; if implied but not confirmed, put under Key Discussion Points instead

Use EXACT headings and order:

# Meeting Summary
# Attendees
# Agenda
# Key Discussion Points
# Decisions Made
# Action Items
# Votes / Motions
# Risks / Blockers
# Next Steps

Attendees:
- Only include names/roles that are clearly mentioned; otherwise write 'Not specified'

Votes / Motions:
- If none clearly stated, write 'None mentioned'
""".strip()  # .strip() removes extra whitespace/newlines at start/end

# user_prompt:
# - includes the transcript and tells the model what to do
# - f-string inserts full_transcript into the message
user_prompt = f"""
Transcript (with timestamps):
{full_transcript}

Now generate the meeting minutes.
""".strip()

# response_stream:
# - chat.completions.create(...) sends messages to the model
# - model=minutes_model_name chooses which Ollama model to use
# - temperature=0 makes output more deterministic (less random)
# - stream=True returns tokens gradually (streaming)
response_stream = ollama_client.chat.completions.create(
    model=minutes_model_name,
    messages=[
        {"role": "system", "content": system_prompt},  # system instructions
        {"role": "user", "content": user_prompt},      # user transcript + request
    ],
    temperature=0,  # low randomness
    stream=True     # stream tokens (we still collect silently)
)

# Collect tokens silently (no console streaming)
generated_minutes = ""  # will store the final minutes text

# Loop through each streamed chunk from the model
for response_chunk in response_stream:
    # response_chunk.choices[0].delta contains the incremental token
    token_data = response_chunk.choices[0].delta

    # getattr(token_data, "content", None):
    # - safely get token_data.content if it exists
    # - otherwise return None
    token_text = getattr(token_data, "content", None)

    # Only append if this chunk contains actual text
    if token_text:
        generated_minutes += token_text  # add token text to final output string

[3/3] Generating meeting minutes via Ollama...



## ==========================================================
## 5) Display Nicely + Save to .md file
## ==========================================================

In [7]:
# Display Markdown nicely in Jupyter Notebook
display(Markdown(generated_minutes))

# Save to a Markdown file in the current working directory
# with open(...) as file:
# - opens file for writing ("w")
# - encoding="utf-8" supports all characters safely

'''

with open("meeting_minutes.md", "w", encoding="utf-8") as file:
    file.write(generated_minutes)  # write the generated text into the file

print("Saved as: meeting_minutes.md")  # confirm saved file name

'''


# Meeting Summary
The council convened to approve previous meeting minutes, share community announcements, and officially recognize Indigenous Peoples Day through a proclamation. Council members highlighted the symbolism of a new logo representing water and cultural confluence, and promoted upcoming events such as a Halloween parade in the Lucky District. Councilman Lopez delivered the Indigenous Peoples Day proclamation, emphasizing respect for Native heritage and the intersection of cultural preservation with environmental stewardship. The session included remarks on inclusivity, pride in cultural identity, and the importance of protecting sacred lands. No formal motions or vote outcomes beyond the approvals were recorded. The meeting concluded without further business items.

# Attendees
- Councilman Lopez  
- Councilman Clark  
- Councilman Martega (request to add name noted)  
- Madam Secretary Raulkaw  
- Mr. President (Chair)  
- Not specified (other council members, staff, public)

# Agenda
1. Approval of October 2 minutes  
2. Council announcements  
3. Presentation of Halloween parade invitation  
4. Reading and adoption of Indigenous Peoples Day proclamation  
5. Discussion of logo symbolism and cultural themes  
6. Open comments from council members and community representatives  

# Key Discussion Points
- **Minutes Approval** – The October 2 minutes were reviewed; no corrections were noted, and they were approved. [00:00]  
- **Halloween Parade Announcement** – Councilman Clark invited members to the first Broadway Halloween parade on October 8, describing activities and encouraging attendance on October 21 at 6 p.m. [02:53‑03:07]  
- **Proclamation Reading** – Councilman Lopez read Proclamation 1127 (2017) honoring Indigenous Peoples Day, detailing Colorado’s tribal heritage and the city’s commitment to cultural recognition. [06:17‑06:33]  
- **Cultural Symbolism** – Speakers praised the new logo’s water motif, linked it to the “confluence” concept, and discussed its relevance to Indigenous heritage and environmental stewardship. [00:00‑13:20]  
- **Community Reflections** – Council members shared personal remarks on pride in cultural identity, the importance of inter‑generational knowledge, and concerns about threats to sacred lands. [08:57‑14:51]  

# Decisions Made
- Approval of the October 2 minutes.  
- Adoption of the Indigenous Peoples Day proclamation (implied by reading and council endorsement).  

# Action Items
- None mentioned  

# Votes / Motions
- None mentioned  

# Risks / Blockers
- Expressed concerns about divisive political climate and potential threats to sacred Indigenous sites, indicating broader cultural and environmental risk factors.  

# Next Steps
- None mentioned  

'\n\nwith open("meeting_minutes.md", "w", encoding="utf-8") as file:\n    file.write(generated_minutes)  # write the generated text into the file\n\nprint("Saved as: meeting_minutes.md")  # confirm saved file name\n\n'

# V2 

In [1]:
# ===============================
# FAST + GOOD: MP3 → Transcript (local Whisper GPU) → Minutes (Ollama OpenAI endpoint)
# Model for minutes: gpt-oss:120b-cloud
# Prereq (run in terminal once):
#   ollama pull gpt-oss:120b-cloud
#   ollama serve
# ===============================

import torch
import librosa
from transformers import pipeline
from openai import OpenAI

# 1) AUDIO PATH
audio_path = r"D:\work\llm_engineering\Data\denver_extract.mp3"

# 2) GPU CHECK
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# 3) LOAD AUDIO (no ffmpeg)
print("\n[1/3] Loading audio...")
audio, _ = librosa.load(audio_path, sr=16000)
print("Audio length (sec):", round(len(audio) / 16000, 1))

# 4) TRANSCRIBE (Whisper on GPU)
print("[2/3] Transcribing with Whisper...")
asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base.en",
    device="cuda",
    return_timestamps=True,
    chunk_length_s=30,
    stride_length_s=(5, 5),
)

res = asr(audio)

def mmss(t):
    if isinstance(t, (tuple, list)):
        t = t[0]
    if t is None:
        return "??:??"
    t = int(t)
    m, s = divmod(t, 60)
    return f"{m:02d}:{s:02d}"

# Build timestamped transcript
if "chunks" in res and res["chunks"]:
    lines = []
    for c in res["chunks"]:
        txt = (c.get("text") or "").strip()
        if not txt:
            continue
        line = f"[{mmss(c.get('timestamp'))}] {txt}"
        if not lines or lines[-1] != line:  # drop consecutive duplicates
            lines.append(line)
    transcript = "\n".join(lines)
else:
    transcript = res.get("text", "")

# Keep request small/fast (adjust if you want)
transcript = transcript[:20000]
print("Transcription done.")

# 5) MEETING MINUTES via OLLAMA (OpenAI-compatible)
print("[3/3] Generating meeting minutes via Ollama...")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"   # any string works for local Ollama
)

system_msg = (
    "You are a professional meeting-minutes generator.\n"
    "Output ONLY the meeting minutes in Markdown.\n"
    "Do NOT copy transcript lines.\n"
    "Do NOT add introductions like 'The meeting summary is...'.\n"
    "Do NOT ask questions.\n"
    "If info is missing, write 'Not specified'.\n"
    "Action items only if explicit; else 'None mentioned'.\n"
    "Use EXACT headings:\n"
    "# Meeting Summary\n# Agenda\n# Key Discussion Points\n# Decisions Made\n# Action Items\n# Risks / Blockers\n"
    "In Key Discussion Points, include inline timestamps like [MM:SS] when relevant."
)

user_msg = f"Transcript (with timestamps):\n{transcript}"

resp = client.chat.completions.create(
    model="gpt-oss:120b-cloud",
    messages=[
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ],
    temperature=0
)

minutes = resp.choices[0].message.content.strip()

print("\n===== MEETING MINUTES =====\n")
print(minutes)


CUDA available: True
GPU: NVIDIA GeForce GTX 1650

[1/3] Loading audio...
Audio length (sec): 900.0
[2/3] Transcribing with Whisper...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> ha

Transcription done.
[3/3] Generating meeting minutes via Ollama...

===== MEETING MINUTES =====

# Meeting Summary
The Denver City Council convened on Monday, October 9, 2017. The meeting included roll call, council announcements, a proclamation honoring Indigenous Peoples Day, and remarks from several council members and community representatives.

# Agenda
- Roll call and quorum verification  
- Council announcements (e.g., Halloween parade invitation)  
- Review of any presentations (none)  
- Proclamation #1127 – Observance of Indigenous Peoples Day  
- Reading and adoption of the proclamation  
- Member remarks and acknowledgments  

# Key Discussion Points
- **[01:09]** Roll call confirmed 11 members present, establishing a quorum.  
- **[01:50]** Councilman Clark announced a Halloween parade on Broadway, scheduled for October 21 at 6 p.m.  
- **[03:16]** Introduction of Proclamation #1127, series 2017, recognizing Indigenous Peoples Day.  
- **[04:07‑04:22]** Councilmember Lopez

#  V3

In [4]:
# ===============================
# Faster + Beginner: MP3 -> Transcript (Whisper GPU) -> Minutes (Ollama)
# ===============================

import torch
import librosa
from transformers import pipeline
from openai import OpenAI

# --------- SETTINGS (edit only these) ----------
audio_path = r"D:\work\llm_engineering\Data\denver_extract.mp3"

WHISPER_MODEL = "openai/whisper-base.en"   # faster option: "openai/whisper-tiny.en"
MINUTES_MODEL = "gpt-oss:20b-cloud"      # was: "gpt-oss:120b-cloud" (very slow)
MAX_TRANSCRIPT_CHARS = 12000               # smaller = faster minutes
# ----------------------------------------------

# 1) GPU / CPU setup
use_cuda = torch.cuda.is_available()
device = 0 if use_cuda else -1

print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

# 2) Load audio
print("\n[1/3] Loading audio...")
audio, _ = librosa.load(audio_path, sr=16000, mono=True)
print("Audio length (sec):", round(len(audio) / 16000, 1))

# 3) Whisper transcription
print("[2/3] Transcribing with Whisper...")

asr = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    device=device,
    torch_dtype=torch.float16 if use_cuda else torch.float32,
    return_timestamps=True,         # keep timestamps (needed for your minutes)
    chunk_length_s=60,              # bigger chunks = less overhead
    stride_length_s=(2, 2),         # smaller stride = less work
)

res = asr(audio)

def mmss(t):
    if isinstance(t, (tuple, list)):  # sometimes timestamp is (start,end)
        t = t[0]
    if t is None:
        return "??:??"
    t = int(t)
    m, s = divmod(t, 60)
    return f"{m:02d}:{s:02d}"

# Build timestamped transcript
if "chunks" in res and res["chunks"]:
    lines = []
    for c in res["chunks"]:
        txt = (c.get("text") or "").strip()
        if not txt:
            continue
        line = f"[{mmss(c.get('timestamp'))}] {txt}"
        if not lines or lines[-1] != line:  # remove consecutive duplicates
            lines.append(line)
    transcript = "\n".join(lines)
else:
    transcript = res.get("text", "")

# Keep it small for faster minutes
transcript = transcript[:MAX_TRANSCRIPT_CHARS]
print("Transcription done. Characters used:", len(transcript))

# 4) Minutes using Ollama (OpenAI compatible)
print("[3/3] Generating meeting minutes via Ollama...")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # any string works for local Ollama
)

system_msg = (
    "You are a professional meeting-minutes generator.\n"
    "Output ONLY the meeting minutes in Markdown.\n"
    "Do NOT copy transcript lines.\n"
    "Do NOT add introductions.\n"
    "Do NOT ask questions.\n"
    "If info is missing, write 'Not specified'.\n"
    "Action items only if explicit; else 'None mentioned'.\n"
    "Use EXACT headings:\n"
    "# Meeting Summary\n# Agenda\n# Key Discussion Points\n# Decisions Made\n# Action Items\n# Risks / Blockers\n"
    "In Key Discussion Points, include inline timestamps like [MM:SS] when relevant."
)

user_msg = f"Transcript (with timestamps):\n{transcript}"

resp = client.chat.completions.create(
    model=MINUTES_MODEL,
    messages=[
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ],
    temperature=0
)

minutes = resp.choices[0].message.content.strip()

print("\n===== MEETING MINUTES =====\n")
print(minutes)


CUDA available: True
GPU: NVIDIA GeForce GTX 1650

[1/3] Loading audio...
Audio length (sec): 900.0
[2/3] Transcribing with Whisper...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Transcription done. Characters used: 7996
[3/3] Generating meeting minutes via Ollama...

===== MEETING MINUTES =====

# Meeting Summary  
A Denver City Council meeting was held on Monday, October 9. The meeting opened with a pledge of allegiance and included the standard quorum check. The council approved the minutes of the previous meeting (October 2), made an announcement for a Broadway Halloween event, and discussed the adoption of a formal proclamation honoring Indigenous Peoples Day. The motion for the proclamation was moved and seconded, and the council adopted the proclamation, which was read in full by Councilman Lopez. No presentations or other communications were made. No additional action items or risks were highlighted.

# Agenda  
1. **Approval of Minutes (October 2)**  
2. **Council Announcements** – Broadway Halloween event  
3. **Presentations** – None  
4. **Communications** – None  
5. **Proclamation of Indigenous Peoples Day** – Motion to adopt and discussion  

# K

# =================== GRADIO Variant =====================

In [6]:
# ==========================================================
# Gradio App: MP3 -> Transcript (Whisper) -> Minutes (Ollama)
# User uploads ANY MP3 and clicks "Create Meeting Minutes"
# Shows Markdown minutes + lets user download meeting_minutes.md
# ==========================================================

import os
import torch
import librosa
import gradio as gr
from transformers import pipeline
from openai import OpenAI


# -----------------------------
# SETTINGS (edit if you want)
# -----------------------------
WHISPER_MODEL = "openai/whisper-small.en"     # faster: "openai/whisper-tiny.en"
MINUTES_MODEL = "gpt-oss:20b-cloud"

MAX_TRANSCRIPT_CHARS = 14000
CHUNK_LENGTH_S = 45
# -----------------------------


# ==========================================================
# 1) Setup: GPU/CPU + load Whisper ONCE (important for speed)
# ==========================================================
use_cuda = torch.cuda.is_available()
device = 0 if use_cuda else -1

dtype_for_whisper = torch.float16 if use_cuda else torch.float32

asr = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL,
    device=device,
    torch_dtype=dtype_for_whisper,
    return_timestamps=True,
    chunk_length_s=CHUNK_LENGTH_S,
    stride_length_s=(2, 2),
)

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


def mmss(t):
    """Convert seconds -> MM:SS (handles tuple timestamps too)."""
    if isinstance(t, (tuple, list)):
        t = t[0]
    if t is None:
        return "??:??"
    t = int(t)
    m, s = divmod(t, 60)
    return f"{m:02d}:{s:02d}"


PRO_SYSTEM_PROMPT = """
You are an expert corporate secretary and council/board meeting scribe.

Your job:
Turn the transcript into clear, accurate, professional meeting minutes.

Output rules (STRICT):
- Output ONLY Markdown minutes (no preface, no explanation, no transcript)
- Do NOT copy transcript lines verbatim; rewrite and summarize
- Keep facts faithful; do NOT invent details
- If something is unclear/missing, write 'Not specified'
- If no action items are explicitly assigned, write 'None mentioned'
- Use blank lines between sections
- Prefer concise bullets; use short paragraphs only where requested

Quality requirements:
- Write a descriptive Meeting Summary (6–8 sentences) explaining context, purpose, and outcomes
- Agenda should be inferred as best as possible from transcript
- Key Discussion Points should be grouped by topic, each with 2–4 bullets
- Each bullet should include a timestamp like [MM:SS] when relevant
- Decisions must be explicit; if implied but not confirmed, put under Key Discussion Points instead

Use EXACT headings and order:

# Meeting Summary
# Attendees
# Agenda
# Key Discussion Points
# Decisions Made
# Action Items
# Votes / Motions
# Risks / Blockers
# Next Steps

Attendees:
- Only include names/roles that are clearly mentioned; otherwise write 'Not specified'

Votes / Motions:
- If none clearly stated, write 'None mentioned'
""".strip()


# ==========================================================
# 2) Main function Gradio will call
# ==========================================================
def create_minutes(mp3_file_path):
    """
    mp3_file_path comes from Gradio after user uploads an mp3.
    We return:
      1) meeting minutes in Markdown (string)
      2) downloadable .md file path
    """
    if mp3_file_path is None:
        return "Please upload an MP3 file.", None

    # ---- Load MP3 into audio array for Whisper ----
    audio, _ = librosa.load(mp3_file_path, sr=16000, mono=True)

    # ---- Whisper transcription ----
    result = asr(audio)

    # Build timestamped transcript
    if "chunks" in result and result["chunks"]:
        lines = []
        last_line = ""
        for c in result["chunks"]:
            txt = (c.get("text") or "").strip()
            if not txt:
                continue
            line = f"[{mmss(c.get('timestamp'))}] {txt}"
            if line != last_line:
                lines.append(line)
                last_line = line
        transcript = "\n".join(lines)
    else:
        transcript = result.get("text", "")

    transcript = transcript[:MAX_TRANSCRIPT_CHARS]

    # ---- Ollama minutes generation (stream -> collect) ----
    user_msg = f"Transcript (with timestamps):\n{transcript}\n\nNow generate the meeting minutes."

    stream = ollama_client.chat.completions.create(
        model=MINUTES_MODEL,
        messages=[
            {"role": "system", "content": PRO_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0,
        stream=True
    )

    minutes_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        token = getattr(delta, "content", None)
        if token:
            minutes_text += token

    # ---- Save to a file for download ----
    out_path = "meeting_minutes.md"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(minutes_text)

    return minutes_text, out_path


# ==========================================================
# 3) Gradio UI
# ==========================================================
with gr.Blocks(title="MP3 to Meeting Minutes") as demo:
    gr.Markdown(
        "# 🎧 MP3 → Meeting Minutes\n"
        "Upload an MP3, then click **Create Meeting Minutes**.\n\n"
        "**Note:** Make sure Ollama is running locally: `ollama serve`"
    )

    mp3_input = gr.File(label="Upload MP3", file_types=[".mp3"])
    btn = gr.Button("Create Meeting Minutes")

    minutes_md = gr.Markdown(label="Meeting Minutes (Markdown)")
    download_file = gr.File(label="Download meeting_minutes.md")

    btn.click(
        fn=create_minutes,
        inputs=[mp3_input],
        outputs=[minutes_md, download_file]
    )

demo.launch()


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
